In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Davis 외부검증 결과를 모아 비교한다. qsar_results/dta_results 모두 davis_selectivity.csv를
# 외부 검증셋으로 계산한 {model}/{dataset}/{target}/davis_metrics.csv로 파일명이 통일되어 있어
# 하나의 로더로 두 소스를 동일하게 처리한다.
qsar_base = '/data/rlawlsgurjh/work/gitlab/CUE/qsar_results'
dta_base = '/data/rlawlsgurjh/work/gitlab/CUE/dta_results'

dataset = 'Kd'
targets = ['PI', 'Ssel', 'WS(alpha=2)', 'RS(k=2)']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 표시 이름 -> (base_dir, base_dir 하위 실제 폴더명)
model_sources = {
    'DeepDTA': (dta_base, 'DeepDTA'),
    'GraphDTA': (dta_base, 'GraphDTA'),
    'TEFDTA': (dta_base, 'TEFDTA'),
    'HMM-DTA': (dta_base, 'HMM-DTA'),
    'ADMET-Print': (qsar_base, 'ADMET-Print'),
    'ADMET-AI': (qsar_base, 'ADMET-AI'),
    'MMFDL': (qsar_base, 'MMFDL'),
    'MMFRL': (qsar_base, 'MMFRL'),
    'CMFE(Ours)': (qsar_base, 'CMFE'),
}

model_order = ['DeepDTA', 'GraphDTA', 'TEFDTA', 'HMM-DTA', 'ADMET-Print', 'ADMET-AI', 'MMFDL', 'MMFRL', 'CMFE(Ours)']

In [6]:
def load_davis_metric(base_dir, model_dir, target):
    """{base_dir}/{model_dir}/Kd/{target}/davis_metrics.csv (davis 외부검증, 단일 행)에서
    mean/std 값을 읽어온다. qsar_results/dta_results 모두 동일 파일명(davis_metrics.csv)으로
    통일되어 있어 분기 없이 하나의 로더로 처리한다."""
    path = os.path.join(base_dir, model_dir, dataset, target, 'davis_metrics.csv')
    if not os.path.exists(path):
        return None
    row = pd.read_csv(path).iloc[0]
    values = {}
    for m in metrics_of_interest:
        if m in row.index:
            values[m] = row[m]
            values[f'{m}_std'] = row.get(f'{m}_std', np.nan)
    return values if values else None


raw_rows = []
for target in targets:
    for display_name, (base_dir, model_dir) in model_sources.items():
        vals = load_davis_metric(base_dir, model_dir, target)
        if vals is not None:
            raw_rows.append({'Dataset': dataset, 'Target': target, 'Model': display_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")

✅ 데이터 로드 완료: 36개의 항목이 수집되었습니다.


In [7]:
def plot_comparison_styled(df_plot, dataset, target_name, metric_name, output_dir):
    """지표 라벨은 지우되, Y축 눈금 수치와 배경은 유지한 최종 버전"""
    df_target = df_plot.set_index('Model').reindex(model_order).dropna(subset=[metric_name]).reset_index()
    if df_target.empty: return

    models = df_target['Model'].tolist()
    values = df_target[metric_name].tolist()
    
    baseline_hatches = ['///', '\\\\', '|||', '---', '+++', 'xxx', '..', '|']
    
    y_min, y_max = min(values), max(values)
    margin = (y_max - y_min) * 0.45 if (y_max - y_min) != 0 else 0.1
    
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(models))
    
    colors = ['lightgray'] * len(models)
    if 'CMFE(Ours)' in models:
        colors[models.index('CMFE(Ours)')] = 'salmon'
    
    bars = ax.bar(x, values, color=colors, edgecolor='black', alpha=0.9, linewidth=1)
    
    for i, bar in enumerate(bars):
        model_name = models[i]
        if model_name != 'CMFE(Ours)':
            bar.set_hatch(baseline_hatches[i % len(baseline_hatches)])
        
        height = bar.get_height()
        # 막대 위 성능 수치 표시 (크기 15, 세로형)
        ax.text(bar.get_x() + bar.get_width()/2., height + (margin * 0.03), 
                f'{height:.3f}', ha='center', va='bottom', fontsize=15, 
                fontweight='bold', rotation=90)
    
    # --- 텍스트 라벨 제거 ---
    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([]) # X축 모델명만 숨김
    
    # --- Y축 수치 및 배경 유지 ---
    # ax.set_yticklabels([]) 을 제거하여 Y축 수치를 복구합니다.
    ax.tick_params(axis='y', labelsize=11) # Y축 숫자 크기 조절
    ax.tick_params(axis='x', which='both', length=0) # X축 눈금선만 깔끔하게 제거
    
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
    
    # 상단 여백 확보
    ax.set_ylim(max(0, y_min - (margin*0.1)), y_max + margin)
    
    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ Y축 수치 복구 및 저장 완료: {save_path}")


In [8]:
plot_base_dir = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results/plots'

output_path = os.path.join(plot_base_dir, 'davis_comparison')
for target in targets:
    df_target = df_total[df_total['Target'] == target]
    if df_target.empty: continue

    for m in metrics_of_interest:
        if m in df_target.columns:
            plot_comparison_styled(df_target, dataset, target, m, output_path)

✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/PI_rmse.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/PI_r2.png


/tmp/ipykernel_1138317/3435840814.py:52: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/PI_pcc.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/Ssel_rmse.png


/tmp/ipykernel_1138317/3435840814.py:52: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/Ssel_r2.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/Ssel_pcc.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/WS(alpha=2)_rmse.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/WS(alpha=2)_r2.png


/tmp/ipykernel_1138317/3435840814.py:52: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/WS(alpha=2)_pcc.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/RS(k=2)_rmse.png
✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/RS(k=2)_r2.png


/tmp/ipykernel_1138317/3435840814.py:52: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ Y축 수치 복구 및 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/davis_comparison/RS(k=2)_pcc.png
